# LangChain con Google AI Studio (Gemini) - Versión Moderna

## Objetivos de Aprendizaje
- Configurar LangChain para usar la API de Google Gemini (AI Studio) con el enfoque moderno.
- Comprender la compatibilidad con la API de OpenAI y las ventajas de las abstracciones actuales.
- Explorar streaming, memoria (`RunnableWithMessageHistory`) y cadenas (`RunnableSequence`).

## ¿Qué es Google AI Studio?
Google AI Studio proporciona acceso gratuito a modelos Gemini (Flash, Pro) con una capa generosa: 1.500 requests/día.

## Instalación de Dependencias

In [ ]:
!pip install langchain langchain-openai langchain-core python-dotenv -q

## Configuración de Credenciales

Obtén tu clave gratuita en [ai.google.dev/gemini-api](https://ai.google.dev/gemini-api).

Puedes usar un archivo `.env` (recomendado) o introducir la clave manualmente con `getpass`.

In [ ]:
import os
from dotenv import load_dotenv
from getpass import getpass

# Cargar variables desde .env (si existe)
load_dotenv()

# Obtener clave (prioridad: entorno > input manual)
API_KEY = os.environ.get("GEMINI_API_KEY")
if not API_KEY:
    print("No se encontró GEMINI_API_KEY en el entorno.")
    API_KEY = getpass("Ingresa tu clave de Google AI Studio: ")
    os.environ["GEMINI_API_KEY"] = API_KEY

# Verificar que la clave no esté vacía
if not API_KEY or len(API_KEY) < 10:
    raise ValueError("❌ La clave parece incorrecta. Debe empezar por 'AIzaSy...'.")

print("✅ Clave configurada correctamente.")
print(f"   (primeros 10 caracteres): {API_KEY[:10]}...")

In [10]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import warnings
warnings.filterwarnings('ignore')

# Endpoint de Google AI Studio compatible con OpenAI
BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

# Modelos disponibles (todos gratuitos)
MODELOS = {
    "flash": "gemini-3.6-flash",
    "flash-2.0": "gemini-2.0-flash",
    "pro": "gemini-1.5-pro",
}

MODELO_DEFECTO = MODELOS["flash"]

def crear_llm(modelo=MODELO_DEFECTO, temperature=0.7, max_tokens=300):
    """Crea un LLM de Gemini con la clave y endpoint correctos."""
    api_key = os.environ.get("GEMINI_API_KEY")
    if not api_key:
        raise ValueError("❌ GEMINI_API_KEY no está configurada en el entorno.")
    return ChatOpenAI(
        base_url=BASE_URL,
        api_key=api_key,
        model=modelo,
        temperature=temperature,
        max_tokens=max_tokens,
    )

llm = crear_llm()

# Prueba básica
print("=== Prueba de Conexión ===")
response = llm.invoke([HumanMessage(content="Hola, responde en una palabra.")])
print(f"Modelo: {MODELO_DEFECTO}")
print(f"Respuesta: {response.content}")
if hasattr(response, 'usage_metadata'):
    print(f"Tokens: {response.usage_metadata.get('total_tokens', 'N/A')}")

=== Prueba de Conexión ===
Modelo: gemini-3.6-flash
Respuesta: Hola.
Tokens: 179


## Streaming en Tiempo Real

LangChain permite recibir la respuesta token por token, mejorando la experiencia de usuario.

In [5]:
def ejemplo_streaming():
    print("=== Streaming (respuesta en tiempo real) ===")
    print("🔹 ", end="")
    for chunk in llm.stream([HumanMessage(content="Escribe un breve haiku sobre la luna.")]):
        print(chunk.content, end="", flush=True)
    print("\n" + "-"*40)

ejemplo_streaming()

=== Streaming (respuesta en tiempo real) ===
🔹 Luz en la noche,
perla
----------------------------------------


## Memoria Conversacional Moderna (RunnableWithMessageHistory)

Esta es la forma recomendada en LangChain >=0.3.0. Gestiona el historial de forma automática y permite múltiples sesiones.

In [6]:
# Definir la plantilla del prompt con historial
prompt_template = ChatPromptTemplate.from_messages([
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

# Crear la cadena: prompt → llm → parser
chain = prompt_template | llm | StrOutputParser()

# Almacén de historiales por sesión
store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Envolver la cadena con gestión de historial
chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

def ejemplo_memoria():
    session_id = "sesion_gemini"
    print("=== Conversación con Memoria (Gemini) ===\n")

    print("[Usuario] Me llamo Carlos y me gusta la astronomía.")
    respuesta1 = chain_with_memory.invoke(
        {"input": "Me llamo Carlos y me gusta la astronomía."},
        config={"configurable": {"session_id": session_id}}
    )
    print(f"[Asistente] {respuesta1}")

    print("\n[Usuario] ¿Qué es lo que más te gusta del espacio?")
    respuesta2 = chain_with_memory.invoke(
        {"input": "¿Qué es lo que más te gusta del espacio?"},
        config={"configurable": {"session_id": session_id}}
    )
    print(f"[Asistente] {respuesta2}")

    print("\n[Usuario] ¿Recuerdas mi nombre y mi interés?")
    respuesta3 = chain_with_memory.invoke(
        {"input": "¿Recuerdas mi nombre y mi interés?"},
        config={"configurable": {"session_id": session_id}}
    )
    print(f"[Asistente] {respuesta3}")

ejemplo_memoria()

=== Conversación con Memoria (Gemini) ===

[Usuario] Me llamo Carlos y me gusta la astronomía.
[Asistente] Closing offer:* Dime en qué te gustaría profundizar o si tienes

[Usuario] ¿Qué es lo que más te gusta del espacio?
[Asistente] Como inteligencia artificial, no tengo sentimientos ni la capacidad

[Usuario] ¿Recuerdas mi nombre y mi interés?
[Asistente] ¡Claro que sí, Carlos! Tu nombre es


## Cadenas (Chains) con RunnableSequence

En lugar de `LLMChain` (obsoleto), utilizamos `RunnableSequence` para mayor flexibilidad.

In [7]:
def ejemplo_cadena():
    # Definir el prompt
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Eres un experto en finanzas. Explica conceptos complejos de forma sencilla."),
        ("human", "{tema}")
    ])

    # Crear la cadena: prompt → llm → parser
    chain = prompt | llm | StrOutputParser()

    tema = "¿Qué es la inflación y cómo afecta a la economía?"
    print("=== Cadena: Explicación Financiera ===")
    respuesta = chain.invoke({"tema": tema})
    print(respuesta)

ejemplo_cadena()

=== Cadena: Explicación Financiera ===
 (The Core Concept):**
        *   *Idea


## Comparación de Modelos Gemini

Todos los modelos de Gemini en AI Studio son gratuitos, así que puedes probarlos sin costo. Ajustamos `max_tokens` a 300 para evitar truncamiento en definiciones.

In [8]:
def probar_modelo(model_id, prompt="Explica qué es la computación cuántica en una frase."):
    try:
        llm_temp = crear_llm(modelo=model_id, max_tokens=300)
        respuesta = llm_temp.invoke([HumanMessage(content=prompt)])
        print(f"🔹 Modelo: {model_id}")
        print(f"   Respuesta: {respuesta.content}")
        if hasattr(respuesta, 'usage_metadata'):
            print(f"   Tokens: {respuesta.usage_metadata.get('total_tokens', 'N/A')}")
        print("-" * 40)
    except Exception as e:
        print(f"❌ Error con {model_id}: {e}")

modelos_gemini = {
    "Gemini 3.6 Flash": "gemini-3.6-flash",
    "Gemini 2.0 Flash": "gemini-2.0-flash",
    "Gemini 1.5 Pro": "gemini-1.5-pro",
}

print("=== Prueba de Modelos Gemini ===\n")
for nombre, modelo_id in modelos_gemini.items():
    probar_modelo(modelo_id)

=== Prueba de Modelos Gemini ===

🔹 Modelo: gemini-3.6-flash
   Respuesta: Refining and Selecting the Best Sentence:**
    *
   Tokens: 310
----------------------------------------
❌ Error con gemini-2.0-flash: Error code: 404 - [{'error': {'code': 404, 'message': 'This model models/gemini-2.0-flash is no longer available. Please update your code to use models/gemini-3.6-flash for the latest features and improvements. We recommend you to use the Interactions API.', 'status': 'NOT_FOUND'}}]
❌ Error con gemini-1.5-pro: Error code: 404 - [{'error': {'code': 404, 'message': 'models/gemini-1.5-pro is not found for API version v1main, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}]


## Conclusión

Este notebook actualizado utiliza las **mejores prácticas de LangChain (>=0.3.0)**:
- **Memoria** con `RunnableWithMessageHistory`.
- **Cadenas** con `RunnableSequence`.
- **Configuración robusta** de credenciales.
- **Metadatos de tokens** para monitorear el consumo.

Google AI Studio sigue siendo una opción excelente y gratuita para experimentar con modelos de última generación.